In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# From QAOA to QAOA-GPT
$
\renewcommand{\ket}[1]{|{#1}\rangle}
\renewcommand{\bra}[1]{\langle{#1}|}
$

---

**What You Will Do:**
* Define the Max Cut problem and understand why it's hard to solve classically
* Learn the core quantum concepts — qubits, superposition, measurement — through interactive visualization
* Translate a graph into a quantum Hamiltonian and implement QAOA with CUDA-Q
* Visualize the barren plateau problem and understand why scaling QAOA is challenging
* Implement one step of Adapt-QAOA and see how **AI** (QAOA-GPT) accelerates quantum circuit design
* Compare standard QAOA, Adapt-QAOA, and QAOA-GPT on circuit depth and solution quality

**Prerequisites:**
* Python and Jupyter notebook familiarity
* No prior quantum computing experience required (we introduce all concepts)
* No prior AI/ML experience required

**Key Terminology:**
* Max Cut
* QAOA (Quantum Approximate Optimization Algorithm)
* Hamiltonian
* Variational Algorithm
* Adapt-QAOA
* QAOA-GPT

**CUDA-Q Syntax:**
* [`cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines the parameterized QAOA circuit
* [`cudaq.sample`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample) — samples measurement outcomes from a quantum circuit
* [`cudaq.observe`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.observe) — computes the expectation value of a spin operator
* Local helpers in [`auxiliary_files/qaoa_helper.py`](auxiliary_files/qaoa_helper.py) — `get_maxcut_hamiltonian`, `maxcut_problem`, and `optimize_maxcut_qaoa`

**Sections:**
* **1.1** Define the Max Cut problem
* **1.2** Define a graph with NetworkX
* **1.3** Classical approaches to Max Cut
* **1.4** From bits to qubits — qubits, the Hamiltonian, and an interactive QAOA explorer
* **1.5** QAOA: the algorithm
* **1.6** Implementing QAOA with CUDA-Q
* **1.7** Challenges in scaling QAOA: circuit depth and barren plateaus
* **1.8** Addressing circuit depth: Adapt-QAOA
* **1.9** Addressing circuit depth with AI: QAOA-GPT
* **1.10** Take-home challenge: compare QAOA variants on a weighted graph

> **Workshop note:** If you are following along without running the notebook, the interactive HTML visualizations in sections 1.4, 1.7, and 1.8 can also be opened directly from the [CUDA-Q Academic Visualization Gallery](https://nvidia.github.io/cuda-q-academic/visualization-gallery.html).


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 12px 15px 12px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00;">&#9889; Resource Guidance:</span>** GPUs are not required for this notebook.
Run the notebook on CPU by default. Sections that use a real quantum processing unit (**QPU**) are **optional** — skip them unless you have hardware access and want to try a live device.

</div>


In [ ]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install dependencies and
## download the QAOA helper used by this notebook.

#!pip install cudaq scipy -q
#!mkdir -p auxiliary_files
#!wget -q https://raw.githubusercontent.com/NVIDIA/cuda-q-academic/main/hybrid-workflows/auxiliary_files/qaoa_helper.py -O auxiliary_files/qaoa_helper.py


> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).


In [ ]:
# Install matplotlib / networkx only if missing
import importlib.util
import subprocess
import sys

for pkg in ("matplotlib", "networkx"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Standard library
import os
import re
from typing import List

# Scientific computing
import numpy as np

# Visualization and graph utilities
import matplotlib.pyplot as plt
import networkx as nx

# CUDA-Q
import cudaq
from cudaq import spin

# Local QAOA implementation
from auxiliary_files.qaoa_helper import (
    get_maxcut_hamiltonian,
    maxcut_problem,
    optimize_maxcut_qaoa,
)



---

## 1.1 Max Cut

**Max Cut** is the problem of splitting a graph's nodes into two groups so that as many edges as possible run between the groups.

In this notebook, such a split is called a *cut*, and the *cut value* is the number of edges that connect nodes in different groups. We use the terms *node* and *vertex* interchangeably, and all graphs here are undirected. The image below shows two cuts of the same graph: one is a maximum cut, and the other is not.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/qaoa-for-max-cut/images/max-cut-illustration.png?raw=true" alt="Illustration of the Max Cut problem: a graph with vertices partitioned into two colored groups. The cut edges as those which cross a dotted line" />

As we encode Max Cut with CUDA-Q, we will use bitstrings to represent group membership: `0` for Group 0 and `1` for Group 1.

---

## 1.2 Defining a graph
For the remainder of this tutorial, we will work with the graph coded in the cell below and will refer to it as `sampleGraph`. We will be using the NetworkX library for the graphs in this tutorial.

In [ ]:
edgeList = [(0,1),(1,2),(2,0),(0,4),(4,5),(5,3),(4,3), (3,6)]
sampleGraph =nx.from_edgelist(edgeList)

nx.set_node_attributes(sampleGraph, values = 0, name = 'color')
sampleGraph.nodes[4]['color'] = 1

node_List : List[int] = list(sampleGraph.nodes())
edges_in_graph = list(sampleGraph.edges())
print('The graph has',sampleGraph.number_of_nodes(),'nodes and', sampleGraph.number_of_edges(), 'edges.')

gray ='#8C8C8C'
green ='#76B900'
color_map = [gray if sampleGraph.nodes[u]['color']==0 else green for u in sampleGraph]

pos = nx.spring_layout(sampleGraph, seed=311)
nx.draw(sampleGraph, with_labels=True, pos = pos, node_color=color_map)
plt.show()

Notice that the coloring above induces a cut with edges $(3,4)$, $(4,5)$, and $(0,4)$. This is not a maximum cut of this graph. The aim of this tutorial is to use a quantum algorithm to find a maximum cut.

In [ ]:
sampleGraph.nodes[4]['color'] = 0

---

## 1.3 Classical approaches to Max Cut

Max Cut is [NP-hard](https://en.wikipedia.org/wiki/NP-hardness), meaning there is no known algorithm that solves it exactly in polynomial time for all graphs. As graphs grow, exact methods quickly become impractical:

- **Brute force** checks every possible split. For a graph with $n$ nodes this means $2^n$ possible assignments — feasible for our 7-node example, but utterly impractical for even moderately sized graphs.
- **Greedy and local-search heuristics** are fast but offer no guarantee on solution quality.
- **Semidefinite programming (SDP) relaxation** — the celebrated [Goemans-Williamson algorithm](https://math.mit.edu/~goemans/PAPERS/maxcut-jacm.pdf) (1995) guarantees a cut whose value is at least **0.878** times the optimal, using a beautiful randomized rounding of an SDP relaxation. This remains the best known worst-case approximation ratio for Max Cut.

Since our `sampleGraph` is small, we can find the exact answer by brute force to use as a benchmark for the quantum algorithm later.

In [ ]:
# Brute-force: check every possible split (only practical for small graphs)
max_cut_value = 0
max_cut_edges = []
subsets = [[]]
for u in sampleGraph.nodes():
    subsets = subsets + [s + [u] for s in subsets]

for subset in subsets:
    cut_val = sum(1 for u, v in sampleGraph.edges()
                  if (u in subset) != (v in subset))
    if cut_val > max_cut_value:
        max_cut_value = cut_val
        group0 = subset

group1 = [u for u in sampleGraph.nodes() if u not in group0]

print(f"Exact max cut value: {max_cut_value}")
print(f"Groups: Group 0={group0}, Group 1={group1}")
print(f"\nWe'll use this as a benchmark for the QAOA result.")

---

## 1.4 A Quantum Approach to Max Cut

We've seen that Max Cut is NP-hard — exact classical solutions don't scale. Quantum computing offers a fundamentally different approach. Before diving into the quantum algorithm, let's build up the key ideas using a simple triangle graph as a running example.

### 1.4.1 Qubits, Amplitudes, and Quantum Circuits

Classical computers store information in **bits** — each bit is either **0** or **1**. A quantum computer uses **qubits** instead. A qubit can be 0, can be 1, or — thanks to a quantum mechanical property called **superposition** — can exist in a blend of both simultaneously.

What does this have to do with Max Cut? Recall that a partition assigns each vertex to one of two groups (Group 0 or Group 1). If we assign one qubit per vertex, then:
- qubit = **0** means "this vertex is in Group 0"
- qubit = **1** means "this vertex is in Group 1"

A single state of all the qubits represents one specific partition. For example, with 3 vertices, the state `010` means: vertex 0 → Group 0, vertex 1 → Group 1, vertex 2 → Group 0.

One important difference between classical and quantum computing is **measurement**. A classical program can inspect any of its bits at any time without changing them. In a quantum computer, the state of the qubits is described by *amplitudes* — one for each possible outcome. An amplitude is a number whose squared magnitude gives the probability of seeing that outcome when we measure. This rule — probability equals amplitude squared — is called **Born's rule**. Measurement collapses the superposition: we get just *one* definite outcome, and the rest of the information is lost.

Here's the key insight: a system of $n$ qubits in superposition holds an amplitude for **every** one of the $2^n$ possible partitions. Our 7-node `sampleGraph` has $2^7 = 128$ possible partitions, so 7 qubits carry amplitudes for all 128 at once. To manipulate those amplitudes, a quantum computer applies **quantum gates** — basic operations that rotate or entangle qubits, much like logic gates (AND, OR, NOT) in a classical computer. A sequence of quantum gates is called a **quantum circuit**. A well-designed circuit exploits quantum interference: by carefully choosing the gates, we can make the amplitudes for good partitions grow and the amplitudes for bad partitions shrink, so that a good partition is what we're most likely to see when we measure

Open the widget in a new tab to see Born's rule and measurement in action on a single qubit. (The 3-D sphere in the widget is called the **Bloch sphere** — a standard way to visualize a single qubit's state, but you don't need to understand it to follow along.)

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Exercise 1:</span>**

<div style="display: flex; align-items: flex-start; gap: 20px;">
<div style="flex: 1;">

1. Drag the **θ** slider away from 0. Watch how the amplitudes (α and β) change and how the probability bars update according to Born's rule.
2. Click **"1 Shot"** a few times — each click is one measurement, collapsing the state to 0 or 1.
3. Try **"100 Shots"** to see the statistics converge to the predicted probabilities.

</div>
<div style="text-align: center; flex: 0 0 210px;">
<a href="https://nvidia.github.io/cuda-q-academic/quick-start-to-quantum/interactive_widget/borns-rule-widget-sampling.html" target="_blank">Open widget in a new tab</a>
</div>
</div>

</div>

### 1.4.2 Seeing a quantum algorithm in action

Before diving into the math, let's see a quantum algorithm working on the simplest interesting graph: a **triangle** (3 nodes, 3 edges). Each node gets one qubit, so there are $2^3 = 8$ possible colorings. (The algorithm is called **QAOA** — Quantum Approximate Optimization Algorithm — and we'll explain exactly how it works in Section 1.5.)

Open the interactive app in a new tab to play the role of the optimizer. Two sliders control the algorithm's behavior:
- **Cost angle (α)** — encodes the graph structure
- **Mixer angle (β)** — controls interference between different colorings

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Exercise 2:</span>**

<div style="display: flex; align-items: flex-start; gap: 20px;">
<div style="flex: 1;">

1. **Start at α = 0, β = 0.** All 8 colorings are equally likely (12.5% each) — the qubits are in a uniform superposition. This is pure random guessing.
2. **Drag the sliders.** Watch the histogram shift — the balance between blue bars (good cuts that cut 2 edges) and gray bars (bad cuts that cut 0 edges) changes with each setting. For some parameter values the good cuts dominate; for others they don't. This redistribution is **quantum interference** at work.
3. **Press "Sample (Measure)."** The superposition collapses to one definite coloring — just as it would on a real quantum computer.
4. **Hover over any bar** to preview that coloring on the triangle.

Don't worry about the **⟨H⟩** badge yet — we'll explain exactly what it means in the next section.

</div>
<div style="text-align: center; flex: 0 0 210px;">
<a href="https://nvidia.github.io/cuda-q-academic/interactive_widgets/triangle-qaoa.html" target="_blank">Open widget in a new tab</a>
</div>
</div>

</div>

### 1.4.3 The Max Cut Hamiltonian — what's happening under the hood

You just watched probabilities shift as you tuned α and β. But what is the quantum circuit actually optimizing? To understand, we need to translate the graph into the language of quantum mechanics. The result is called a **Hamiltonian** — a scoring function that the quantum computer evaluates directly on its qubits.

For Max Cut, the Hamiltonian is:

$$H = \frac{1}{2}\sum_{(u,v)\in E} (Z_u Z_v - I)$$

Let's unpack this using the triangle you just explored.

#### What is $Z$?

$Z$ (the Pauli-Z operator) checks a qubit's value:
- If the qubit is in state **0**, $Z$ returns **+1**
- If the qubit is in state **1**, $Z$ returns **−1**

That's it — $Z$ converts our 0/1 partition labels into +1/−1 values.

#### What does $Z_u Z_v$ tell us?

For an edge $(u, v)$, the product $Z_u Z_v$ reveals whether the two endpoints are in the **same** group or **different** groups:

| Vertex $u$ | Vertex $v$ | $Z_u$ | $Z_v$ | $Z_u Z_v$ | Same group? | Edge cut? |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 0 | 0 | +1 | +1 | **+1** | Yes | No |
| 0 | 1 | +1 | −1 | **−1** | No | **Yes** |
| 1 | 0 | −1 | +1 | **−1** | No | **Yes** |
| 1 | 1 | −1 | −1 | **+1** | Yes | No |

When the edge **is** cut: $\;\frac{1}{2}(Z_u Z_v - 1) = \frac{1}{2}(-1 - 1) = -1$

When the edge **is not** cut: $\;\frac{1}{2}(Z_u Z_v - 1) = \frac{1}{2}(+1 - 1) = 0$

So each cut edge contributes $-1$ to $H$, each uncut edge contributes $0$. **Minimizing** $H$ is the same as **maximizing** cut edges!

#### Worked example: hover over bitstring `100` in the QAOA Max Cut Explorer app

Go back to the QAOA Max Cut Explorer app and hover over the bar labeled **`100`**. You'll see node 0 turn green (Group 1) and nodes 1, 2 stay gray (Group 0). Two of the three edges are cut. Let's verify with the formula:

| Edge | $Z_u$ | $Z_v$ | $Z_u Z_v$ | $\frac{1}{2}(Z_u Z_v - 1)$ | Cut? |
|:-:|:-:|:-:|:-:|:-:|:-:|
| (0, 1) | −1 | +1 | −1 | −1 | ✓ |
| (1, 2) | +1 | +1 | +1 | 0 | ✗ |
| (0, 2) | −1 | +1 | −1 | −1 | ✓ |

$$H = (-1) + 0 + (-1) = -2$$

This partition cuts **2 out of 3** edges — the best you can do on a triangle. (Try hovering over `000` or `111` — those cut zero edges, giving $H = 0$.)

#### The expectation value ⟨H⟩ — what the green badge shows

Now look at the **⟨H⟩** badge in the QAOA Max Cut Explorer app. In quantum computing, we work with a superposition of *all* partitions simultaneously. The **expectation value** $\langle H \rangle$ is the weighted average of $H$ over all possible measurement outcomes:

$$\langle H \rangle = \sum_{i} P(\text{state } i) \times H(\text{state } i)$$

For the triangle, each of the 8 bitstrings has $H = 0$ (the two "all same color" states) or $H = -2$ (the six good-cut states). At α = β = 0, all 8 are equally likely (12.5% each):

$$\langle H \rangle = 2 \times 0.125 \times 0 + 6 \times 0.125 \times (-2) = -1.50$$

Go check — the badge should show exactly −1.50 at the default settings. As you find better parameters, ⟨H⟩ drops toward −2.00 — the quantum circuit is converging on a max cut.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Exercise 3:</span>**

<div style="display: flex; align-items: flex-start; gap: 20px;">
<div style="flex: 1;">

Use the sliders in the QAOA Max Cut Explorer applet to push ⟨H⟩ all the way to −2.00. What combination of α and β achieves the optimum? How does the probability distribution change as you approach it?

</div>
<div style="text-align: center; flex: 0 0 210px;">
<a href="https://nvidia.github.io/cuda-q-academic/interactive_widgets/triangle-qaoa.html" target="_blank">Open widget in a new tab</a>
</div>
</div>

</div>

### 1.4.4 Key quantum concepts

The [QAOA Max Cut Explorer app](https://nvidia.github.io/cuda-q-academic/interactive_widgets/triangle-qaoa.html) illustrates the core quantum ideas that power the algorithm:

- **Superposition.** With α = β = 0, the quantum circuit puts all qubits into an equal superposition — every possible partition is equally likely. On a real quantum computer, superposition is what lets us represent all $2^n$ partitions at once.

- **Parameterized quantum circuit.** The α and β sliders control the circuit's behavior. Different parameter values produce different probability distributions over the 8 possible bitstrings. The circuit itself is fixed in structure; only the parameters change.

- **Quantum interference.** As you tune the parameters, the quantum amplitudes for different states add up constructively (boosting good partitions) or destructively (suppressing bad ones). This is fundamentally different from classical randomness — it's what gives quantum computing its potential advantage.

- **Expectation value.** The ⟨H⟩ badge shows the weighted average of the Hamiltonian over all possible measurement outcomes. It's the single number that the classical optimizer uses to judge how good the current parameters are. More negative = better. When ⟨H⟩ reaches its minimum (−2 for the triangle), the circuit has found parameter values where measurements are overwhelmingly likely to produce a max cut.

- **Measurement.** A quantum computer can't hand you the full probability distribution directly. When you press "Sample," the superposition collapses to a single bitstring — one specific partition. Better parameters mean a higher probability of measuring a good cut.

In practice, you don't tune the sliders by hand. A **classical optimizer** (a conventional algorithm on a regular computer) adjusts α and β automatically, trying to minimize ⟨H⟩. This **hybrid quantum-classical loop** is the essence of the algorithm — called **QAOA** — which we'll describe formally in Section 1.5.

---

## 1.5 QAOA: The Algorithm

Now that we have an intuition for what QAOA does, let's look at the algorithm more precisely.

**QAOA** is a **variational algorithm** — it uses a classical optimizer to tune the parameters of a quantum circuit until the circuit produces states that minimize the Hamiltonian (i.e., maximize the cut).

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/qaoa-for-max-cut/images/QAOA-flowchart.png?raw=true" alt="Flowchart of the QAOA hybrid quantum-classical loop: input graph and Hamiltonian feed into an optimization loop where a quantum circuit is executed, the cost function is evaluated, and a classical optimizer updates parameters until convergence, then the final circuit is sampled to produce a Max Cut partition." />

The flowchart above illustrates the hybrid classical-quantum loop:
1. A classical optimizer proposes parameter values (the angles α and β you were adjusting manually in the widget).
2. A quantum circuit, parameterized by those values, prepares a quantum state.
3. The expectation value ⟨H⟩ is estimated — this is the same quantity you watched change in the widget (the `observe` step).
4. The classical optimizer uses ⟨H⟩ as feedback to propose new, better parameters, driving it more negative.
5. Once converged, the final circuit is sampled (the `sample` step) to read out a candidate solution.

### 1.5.1 Inside the QAOA circuit

The diagram below shows the general structure of a QAOA circuit — notice the similarity to what you saw in the triangle widget:

* For each vertex in the graph, there is one qubit in the circuit.
* **Hadamard gates (H)** on the left initialize every qubit into superposition, just like setting α = β = 0 in the widget. (A note on notation: the letter *H* is used for both the Hadamard gate and the Hamiltonian. This is an unfortunate collision that is standard in the field. Context makes it clear which is meant: *H* as a gate appears in circuit diagrams, while *H* as the Hamiltonian appears in cost expressions like $\langle H \rangle$.)
* The circuit then applies repeated *layers* of a **problem kernel** and a **mixer kernel**.
    * The **problem kernel** (blue) encodes the Hamiltonian — each edge in the graph produces quantum gates. This is the part of the computation dependent on the α parameter.
    * The **mixer kernel** (green) applies rotation gates that create quantum interference. This is the part of the computation dependent on the β parameter.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/qaoa-for-max-cut/images/qaoa-circuit-layers.png?raw=true" alt="Diagram of a QAOA circuit: Hadamard gates initialize all qubits into superposition, followed by repeated layers of a problem kernel (blue, encoding the graph Hamiltonian) and a mixer kernel (green, applying Rx rotations)." />

Looking more closely at the problem kernel: for each edge $(u,v)$ in the graph, a controlled-$X$ gate is applied between the corresponding qubits, followed by a parameterized $Z$-rotation and another controlled-$X$ gate.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/qaoa-for-max-cut/images/qaoa-problem-kernel.png?raw=true" alt="Detail of the QAOA problem kernel for a 5-node graph: for each edge, a CNOT gate connects the two qubits, a parameterized Rz rotation is applied, and a second CNOT completes the entangling operation. The edge (1,2) is highlighted." />

More QAOA layers (higher $p$) give the optimizer more knobs to turn, potentially finding better solutions — but at the cost of deeper circuits and more parameters to optimize.

---

## 1.6 Implementing QAOA with CUDA-Q

With CUDA-Q we need three pieces: the Max Cut Hamiltonian, a parameterized quantum circuit (kernel), and a classical optimization loop. For most of this lesson those live in [`auxiliary_files/qaoa_helper.py`](auxiliary_files/qaoa_helper.py), so the notebook can stay focused on the graph problem and the results.

Later in this section we'll also write the same p=1 circuit *explicitly* — gate by gate — so you can see how the helper's cost and mixer layers are built, and so Section 1.8 can reuse that kernel when it adds one adaptive operator.

If you'd like a longer walkthrough of building QAOA from scratch, check out the [QAOA for Max Cut notebook on CUDA-Q Academic](https://nvidia.github.io/cuda-q-academic/learningpath.html?custom=eyJ0aXRsZSI6IlFBT0EgZm9yIE1heCBDdXQiLCJkZXNjIjoiUUFPQSBmb3IgTWF4IEN1dCB3aXRoIENVREEtUSIsIm5icyI6W3sibmFtZSI6Ik1heCBDdXQgZm9yIFFBT0EiLCJjb2xhYiI6Imh0dHBzOi8vY29sYWIucmVzZWFyY2guZ29vZ2xlLmNvbS9naXRodWIvTlZJRElBL2N1ZGEtcS1hY2FkZW1pYy9ibG9iL21haW4vcWFvYS1mb3ItbWF4LWN1dC8wMV9NYXgtQ3V0LXdpdGgtUUFPQS5pcHluYiIsImdoIjoiaHR0cHM6Ly9naXRodWIuY29tL05WSURJQS9jdWRhLXEtYWNhZGVtaWMvYmxvYi9tYWluL3Fhb2EtZm9yLW1heC1jdXQvMDFfTWF4LUN1dC13aXRoLVFBT0EuaXB5bmIiLCJ0eXBlIjoibm90ZWJvb2sifV19).


### 1.6.1 The Max Cut Hamiltonian in code

The local `get_maxcut_hamiltonian` helper generates the cost Hamiltonian directly from our NetworkX graph using CUDA-Q spin operators. It translates each graph edge, including an optional edge weight, into the Pauli-Z operators discussed in Section 1.4.2.

In [ ]:
H = get_maxcut_hamiltonian(sampleGraph)

print("Max Cut Hamiltonian for sampleGraph:")
print(H)

### 1.6.2 Running the QAOA optimization

The local `optimize_maxcut_qaoa` helper builds the circuit layers and runs the quantum-classical loop. We provide:
1. The graph and number of QAOA layers (p).
2. The initial parameters for the optimizer.
3. The optimizer type (COBYLA).

A quick note on naming: the interactive widgets earlier called the cost and mixer angles **α** and **β**. In the code (and in most of the QAOA literature) those same angles are **γ** (gamma) and **β**. They are the same two parameters — only the symbol for the cost angle changes.

Let's wrap this into a notebook-level function so we can reuse the same optimization loop as we explore Adapt-QAOA later.


In [ ]:
def qaoa_for_graph(G, layer_count, seed):
    """Run QAOA to approximate the max cut of a graph with CUDA-Q.

    Parameters
    ----------
    G: networkX graph
        Problem graph whose max cut we aim to approximate
    layer_count : int
        Number of layers in the QAOA circuit
    seed : int
        Random seed for reproducibility of results

    Returns
    -------
    str
        Binary string representing the most probable coloring found by QAOA
    """
    parameter_count = 2 * layer_count
    np.random.seed(seed)
    cudaq.set_random_seed(seed)
    initial_parameters = np.random.uniform(-np.pi, np.pi, parameter_count)

    opt_value, opt_params, opt_config = optimize_maxcut_qaoa(
        G,
        layer_count,
        initial_parameters,
        optimizer="cobyla",
    )

    print("Optimal parameters = ", opt_params)
    print("Most probable outcome = ", opt_config.most_probable())

    return str(opt_config.most_probable())

Let's run QAOA with a single layer (`p=1`) on `sampleGraph` and see what cut it finds. With only one layer, QAOA is a coarse approximation: the most probable bitstring is often a good cut, but the full distribution still puts substantial probability on suboptimal ones.

The `seed` parameter controls both the classical optimizer's random starting point and the simulator's internal random number generator, making results fully reproducible. Execute the cell below to run QAOA, interpret the resulting bitstring as a node coloring, and compute the cut value.


In [ ]:
# Run this notebook on CPU by default using 'qpp-cpu'. Real-QPU sections later are optional.
# Since this quantum circuit is not too large, we can run it on a CPU.  For a full list of available targets, see:
# https://nvidia.github.io/cuda-quantum/latest/using/backends/backends.html
cudaq.set_target('qpp-cpu')  

result = qaoa_for_graph(sampleGraph, layer_count=1, seed=110)

graphColors = [int(i) for i in result]
nodes = sorted(list(nx.nodes(sampleGraph)))

qaoa_cut_value = 0
cut_edges = []

for u, v in sampleGraph.edges():
    indexu = nodes.index(u)
    indexv = nodes.index(v)
    if graphColors[indexu] != graphColors[indexv]:
        qaoa_cut_value += 1
        cut_edges.append((u, v))

group0 = []
group1 = []
for u in sampleGraph.nodes():
    indexu = nodes.index(u)
    if graphColors[indexu] == 0:
        group0.append(u)
        sampleGraph.nodes[u]['color'] = 0
    else:
        group1.append(u)
        sampleGraph.nodes[u]['color'] = 1

print(f'\nQAOA (p=1) cut value: {qaoa_cut_value}  (optimal = {max_cut_value})')
print(f'Groups: Group 0={group0}, Group 1={group1}')
if qaoa_cut_value == max_cut_value:
    print('QAOA found the optimal max cut!')
else:
    print(f'QAOA found a suboptimal cut. With p=1, this can happen — try increasing p for more reliable results.')

In [ ]:
max_cut_color_map = [gray if sampleGraph.nodes[u]['color']==0 else green for u in sampleGraph]

nx.draw_networkx_edges(
    sampleGraph,
    pos,
    edgelist=cut_edges,
    width=8,
    alpha=0.5,
    edge_color=green,
)
nx.draw(sampleGraph, with_labels=True, pos = pos, node_color=max_cut_color_map)
plt.show()

**Did QAOA find the optimal cut?** Compare the cut value above to the brute-force optimum of 6 from Section 1.3. On this small graph, `p=1` often *peaks* on an optimal bitstring — but that single sample is a thin scorecard.

With only **two parameters** (one cost angle γ and one mixer angle β), a single QAOA layer can steer probability toward good cuts, yet the distribution still spreads across many bitstrings. Three things to keep in mind:

1. **Expectation value ⟨H⟩ vs. one sample.** The optimizer minimizes ⟨H⟩, which averages over *all* outcomes. A most-probable bitstring of cut 6 can coexist with an approximation ratio well below 1 (for seed 110, ⟨H⟩ ≈ −4.47 while the optimum is −6).
2. **Landscape structure.** At `p=1` the landscape is shallow (we'll visualize this in Section 1.7), so different random starts can land in different basins — some with better ⟨H⟩ than others.
3. **Sampling noise.** Several near-tied bitstrings sit at the top of the distribution, so which one `most_probable()` returns can bounce from run to run even when the parameters are fixed.

That gap between "the peak bitstring looks good" and "most shots still miss the optimum" is exactly why deeper circuits (higher `p`), Adapt-QAOA (Section 1.8), and QAOA-GPT (Section 1.9) exist — each aims for more reliable probability mass on high-quality cuts, not just a lucky top sample.


### 1.6.3 Writing the p=1 kernel explicitly

An alternative method of coding QAOA is to define the kernel explicitly, gate by gate, instead of letting the helper build it. The kernel below constructs the same p=1 circuit the helper optimizes — one cost layer built from the graph's edges, then one mixer layer — so we can reuse the optimal parameters found above and pass the circuit to `cudaq.sample` to identify a cut that approximates the max cut.

The kernel also ends with one empty `exp_pauli` slot. We leave it as the identity here, and in Section 1.8 we fill it with a single adaptive operator — that one slot is the whole difference between standard QAOA and the first step of Adapt-QAOA.


In [ ]:
# This kernel must build its cost layer exactly the way
# auxiliary_files/qaoa_helper.py does — per edge: CNOT, rz(2*gamma*weight), CNOT
# — so that the parameters optimized below describe this same circuit.
@cudaq.kernel
def kernel_p1_plus_operator(qubit_count: int,
                            edge_sources: list[int],
                            edge_targets: list[int],
                            edge_weights: list[float],
                            gamma: float,
                            beta: float,
                            extra_op: list[cudaq.pauli_word],
                            extra_theta: float):
    qubits = cudaq.qvector(qubit_count)
    h(qubits)
    # Cost layer
    for edge in range(len(edge_sources)):
        source = edge_sources[edge]
        target = edge_targets[edge]
        x.ctrl(qubits[source], qubits[target])
        rz(2.0 * gamma * edge_weights[edge], qubits[target])
        x.ctrl(qubits[source], qubits[target])
    # Mixer layer
    for qubit in range(qubit_count):
        rx(2.0 * beta, qubits[qubit])
    # One additional operator — the identity until Section 1.8
    exp_pauli(extra_theta, qubits, extra_op[0])


# The edge arrays and the Hamiltonian share one qubit numbering (nodes sorted).
n_q, edge_src, edge_tgt, edge_w, H_hw = maxcut_problem(sampleGraph)

edges_in_graph = list(sampleGraph.edges())
node_idx_hw = {n: i for i, n in enumerate(sorted(sampleGraph.nodes()))}
identity_word = 'I' * n_q  # leaves the extra-operator slot empty

# Re-extract the optimized p=1 parameters (same optimization as above)
np.random.seed(110)
cudaq.set_random_seed(110)
init_params_hw = np.random.uniform(-np.pi, np.pi, 2)
opt_value_p1, opt_params_p1, _ = optimize_maxcut_qaoa(
    sampleGraph,
    1,
    init_params_hw,
    optimizer="cobyla",
)
gamma_p1 = float(opt_params_p1[0])
beta_p1 = float(opt_params_p1[1])

print(f"Optimized p=1 parameters: \u03b3 = {gamma_p1:.6f},  \u03b2 = {beta_p1:.6f}")
print(f"Optimized \u27e8H\u27e9 = {opt_value_p1:.4f}")


In [ ]:
shots = 5000

def cut_value_from_bitstring(bitstring):
    """Compute the cut value for a given bitstring coloring."""
    return sum(1 for u, v in edges_in_graph
               if bitstring[node_idx_hw[u]] != bitstring[node_idx_hw[v]])

def probability_of_optimal_cut(counts):
    """Fraction of sampled shots whose bitstring achieves the optimal cut."""
    total = sum(count for _, count in counts.items())
    return sum(count for bs, count in counts.items()
               if cut_value_from_bitstring(bs) == max_cut_value) / total

cudaq.set_target('qpp-cpu')

# This kernel should reproduce the cost the helper reported for the same
# parameters. If it does not, the two cost layers have drifted apart and the
# sampled bitstrings below would be meaningless.
kernel_cost = cudaq.observe(
    kernel_p1_plus_operator, H_hw, n_q, edge_src, edge_tgt, edge_w,
    gamma_p1, beta_p1, [identity_word], 0.0
).expectation()
print(f"\u27e8H\u27e9 from the helper: {opt_value_p1:.6f}"
      f"   from this kernel: {kernel_cost:.6f}")
if abs(kernel_cost - opt_value_p1) > 1e-6:
    print("WARNING: the two cost layers disagree \u2014 check the kernel convention.")

# \u2550\u2550 Sample in simulation (reference) \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
counts_sim = cudaq.sample(
    kernel_p1_plus_operator, n_q, edge_src, edge_tgt, edge_w,
    gamma_p1, beta_p1, [identity_word], 0.0, shots_count=shots
)

sim_bs = counts_sim.most_probable()
sim_cut = cut_value_from_bitstring(sim_bs)
sim_p_optimal = probability_of_optimal_cut(counts_sim)
print(f"Simulation  \u2014 most probable: {sim_bs}  (cut = {sim_cut} / {max_cut_value})")
print(f"            approximation ratio = {-kernel_cost / max_cut_value:.4f},  "
      f"P(optimal cut) = {sim_p_optimal:.1%}")

---

## 1.7 Challenges in scaling QAOA

QAOA works beautifully on small graphs like our 7-node example. But as we try to tackle larger, real-world problems, a central difficulty emerges: **circuit depth**.

The **circuit depth** is the number of sequential gate layers in a quantum circuit — it determines how long the qubits must stay coherent. Deeper circuits (more QAOA layers) can lead to **barren plateaus** — regions where the cost landscape becomes exponentially flat, making it nearly impossible for the classical optimizer to find a good direction. **Adapt-QAOA** (Section 1.8) and **QAOA-GPT** (Section 1.9) attack this problem from different angles.

Let's visualize the barren plateau problem first, then tackle it with Adapt-QAOA and QAOA-GPT.

### 1.7.1 The QAOA optimization landscape


In Exercise 2, you manually tuned α and β to find good cuts on the triangle. The classical optimizer inside QAOA does the same thing — but it navigates a **cost landscape**: the expectation value of the Hamiltonian plotted as a function of the parameters. Open the widget in a new tab to visualize this landscape.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Exercise 4:</span>**

<div style="display: flex; align-items: flex-start; gap: 20px;">
<div style="flex: 1;">

Explore the landscape, then use the **"Barren plateau"** slider to increase the problem size. What happens to the valleys and gradients? At what point does the landscape become too flat for an optimizer to find a good direction?

</div>
<div style="text-align: center; flex: 0 0 210px;">
<a href="https://nvidia.github.io/cuda-q-academic/interactive_widgets/qaoa-landscape.html" target="_blank">Open widget in a new tab</a>
</div>
</div>

</div>

**What to look for:**
- **Clear valleys** correspond to good parameter values — the optimizer can follow the gradient downhill.
- As the problem size increases, the landscape flattens: the valleys disappear, gradients vanish, and the optimizer has no signal to follow. This is the **barren plateau** problem — and it's why simply adding more QAOA layers doesn't always help.

We'll return to barren plateaus when we discuss Adapt-QAOA (Section 1.8) and QAOA-GPT (Section 1.9), which offer strategies for navigating or avoiding them entirely.


---

## 1.8 Addressing circuit depth: Adapt-QAOA

In Section 1.7, we saw how deeper QAOA circuits can run into **barren plateaus** — the cost landscape flattens and the optimizer loses its signal. One approach to this problem is to let the algorithm *build its own circuit* rather than using a fixed template.

Think of Adapt-QAOA like a **greedy algorithm** for building circuits: instead of using a pre-made template (the fixed layers of standard QAOA), we look at our current state and ask, *"Which single gate would help the most right now?"* Then, we add it, re-tune, and repeat.

In standard QAOA, the circuit structure is fixed. Before we run the program, we decide in advance how many layers to use, and each layer applies the same problem and mixer operators. The classical optimizer only tunes the *parameters*. **Adapt-QAOA** takes a different approach: it grows the circuit one operator at a time, always choosing the operator that improves the cost function the most:

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/images/Adapt-QAOA-flowchart.png?raw=true" alt="Flowchart of Adapt-QAOA: evaluate candidate operators, choose the one with the largest gradient, optimize the circuit parameters, and repeat until convergence." />

The flowchart above shows the adaptive loop that incrementally builds the circuit based on which operator is most useful at each step.

1. Start with a **pool** of candidate operators (different mixer gates — think of this as a menu of possible next moves).
2. Compute the **gradient** of the cost function with respect to adding each operator (i.e., ask "how much would this gate help if I turned it on just a little?")
3. Select the operator with the **largest gradient** and append it to the circuit.
4. Re-optimize all parameters.
5. Repeat until the gradients are small enough (convergence).

But what does "largest gradient" actually mean? Open the widget in a new tab to make this concrete. Imagine you've just run one layer of QAOA and you want to add one more operator to improve the result. You have several candidates, and each one changes the cost differently depending on how strongly you apply it (a parameter θ). The **gradient** is just the slope of the cost curve at θ = 0 (e.g., how fast the cost changes when you first turn on that operator). A steep slope means big changes in the cost function (i.e., cut value); a flat slope means the operator barely has an effect.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Exercise 5:</span>**

<div style="display: flex; align-items: flex-start; gap: 20px;">
<div style="flex: 1;">

One QAOA layer has already been applied to a triangle graph. Your job: decide which operator to add next.

1. Click each operator button and drag the **θ slider** away from 0 — watch the C(θ) curve change shape.
2. Check the **gradient readout**: which operator has the steepest downward slope at θ = 0?
3. That's your pick. Click **"Add operator"** to append it to the circuit.

</div>
<div style="text-align: center; flex: 0 0 210px;">
<a href="https://nvidia.github.io/cuda-q-academic/interactive_widgets/operator-gradient.html" target="_blank">Open widget in a new tab</a>
</div>
</div>

</div>



The key insight is that by *adapting* the circuit structure, Adapt-QAOA can often find better solutions with *fewer* total gates than a fixed-depth QAOA — or achieve the same quality with shallower circuits. This directly combats barren plateaus: instead of blindly stacking layers and hoping the optimizer can navigate a flat landscape, Adapt-QAOA only adds operations that demonstrably improve the solution.

In Section 1.8.1 below, you'll try this yourself on our 7-node `sampleGraph` — picking one operator to add to the p=1 circuit, measuring the improvement in cost and approximation ratio, and optionally testing the result on the IQM QPU.


The previous widget showed how individual operator gradients guide the selection of the next gate. Open the interactive demo in a new tab to put the **full ADAPT-QAOA loop** in your hands for a slightly larger graph: you'll build the ansatz layer-by-layer — selecting the mixer with the largest gradient from a pool, optimizing parameters, and watching the cost converge — while a standard QAOA baseline runs side by side for comparison.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Exercise 6:</span>**

<div style="display: flex; align-items: flex-start; gap: 20px;">
<div style="flex: 1;">

1. In the **ADAPT-QAOA** panel, click the mixer button with the largest gradient (green bar) to add it as the next layer.
2. Click **"Minimize (γ, β)"** to optimize the parameters for the circuit so far.
3. Repeat until the algorithm converges (gradient norm drops below threshold).
4. Click **"Run QAOA (p=4)"** to see how standard QAOA compares on the same 5-node Max-Cut instance.
5. Compare the two cost curves: how quickly does each approach reach the optimum?

</div>
<div style="text-align: center; flex: 0 0 210px;">
<a href="https://nvidia.github.io/cuda-q-academic/interactive_widgets/adapt-qaoa.html" target="_blank">Open widget in a new tab</a>
</div>
</div>

</div>

**Further reading:**
- Zhu *et al.*, ["An adaptive quantum approximate optimization algorithm for solving combinatorial problems on a quantum computer"](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.4.033029), *Phys. Rev. Research* **4**, 033029 (2022)


### 1.8.1 Extending the p=1 circuit — your first step toward Adapt-QAOA

The previous widget illustrated the core idea on a 5-vertex graph: different operators have different gradients, and the steepest one improves the cost fastest. Now let's apply exactly this strategy to our 7-node `sampleGraph` with CUDA-Q simulations — and then test the result on **IQM quantum hardware**.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Exercise 7:</span>**

1. Start from the optimized QAOA p=1 circuit (the parameters we found in Section 1.6).
2. Evaluate a pool of candidate operators by computing each one's **gradient** — how steeply the cost drops when you first turn it on (θ = 0 → small θ).
3. Pick the winner and optimize its strength parameter θ.
4. Optionally run the improved circuit on a real **QPU** (quantum processing unit) and compare to the plain p=1 baseline.

</div>

This is precisely one iteration of the Adapt-QAOA loop done by hand. We reuse `kernel_p1_plus_operator` from Section 1.6 — the fixed p=1 layer using the parameters we already optimized, followed by one extra operator — except that this time we fill in the operator slot instead of leaving it as the identity.


We define a pool of candidate operators — a mix of single-qubit rotations and two-qubit entangling gates acting on various edges of the graph. For each candidate, we compute the **gradient** at θ = 0 using finite differences:

$$\text{gradient} \approx \frac{E(\varepsilon) - E(-\varepsilon)}{2\varepsilon}$$

This works for any operator — single Pauli strings, sums of Paulis, or any other generator you can express with `exp_pauli`. (For single Pauli generators there is also an exact method called the parameter shift rule, but finite differences are more general and simpler to apply across a mixed operator pool.)

A large negative gradient means the cost drops steeply when the operator is first turned on — exactly what we saw on the C(θ) plot in the widget. The operator with the steepest slope is the one Adapt-QAOA would select.

> **Optional extension:** Try adding your own operators to the pool! Any 7-character Pauli string (using `I`, `X`, `Y`, `Z`) defines a valid operator on the 7-qubit system.

In [ ]:
cudaq.set_target('qpp-cpu')

candidate_operators = [
    ('XIIIIII', 'X₀  (single-qubit)'),
    ('IIIXIII', 'X₃  (single-qubit)'),
    ('IIIIIIX', 'X₆  (single-qubit)'),
    ('YYIIIII', 'Y₀Y₁  (edge 0–1)'),
    ('IIIYYII', 'Y₃Y₄  (edge 3–4)'),
    ('IIIIYYI', 'Y₄Y₅  (edge 4–5)'),
    ('IIIZIIY', 'Z₃Y₆  (edge 3–6)'),
    # YOUR OPERATORS HERE — try adding more candidates!
]

eps = 0.005

base_cost = cudaq.observe(
    kernel_p1_plus_operator, H_hw, n_q,
    edge_src, edge_tgt, edge_w, gamma_p1, beta_p1, [identity_word], 0.0
).expectation()

print(f"QAOA p=1 baseline cost: {base_cost:.4f}  (optimal = -{max_cut_value})")
print(f"\n{'Operator':<28} {'Gradient':>10}  {'|Gradient|':>10}")
print('=' * 52)

gradient_results = []
for pauli_str, label in candidate_operators:
    E_plus = cudaq.observe(
        kernel_p1_plus_operator, H_hw, n_q,
        edge_src, edge_tgt, edge_w, gamma_p1, beta_p1, [pauli_str], eps
    ).expectation()

    E_minus = cudaq.observe(
        kernel_p1_plus_operator, H_hw, n_q,
        edge_src, edge_tgt, edge_w, gamma_p1, beta_p1, [pauli_str], -eps
    ).expectation()

    grad = (E_plus - E_minus) / (2 * eps)
    gradient_results.append((pauli_str, label, grad))
    print(f"  {label:<26} {grad:>+10.4f}  {abs(grad):>10.4f}")

best_op_str, best_op_label, best_grad = max(gradient_results, key=lambda x: abs(x[2]))
print(f"\n→ Steepest gradient: {best_op_label}  (gradient = {best_grad:+.4f})")

> **Questions:**
> - Which operator has the steepest gradient? Is it a single-qubit or two-qubit operator?
> - The standard QAOA mixer already applies single-qubit X rotations on every qubit. If a single-qubit operator still wins here, what might that tell you about the p=1 parameters? If a two-qubit operator wins, what does that suggest about the graph structure?
> - Does the winning operator correspond to an edge in the graph? What does that tell you about which parts of the graph the p=1 circuit struggled with?

Now let's optimize the strength θ of the best operator by scanning across a range of values. We'll plot C(θ) — the same curve you explored in the widget — and find the θ that minimizes the cost.

One caution on how we measure progress: on this graph the top few bitstrings are separated by only a fraction of a percent, so the single most probable bitstring is a noisy scorecard — it can change from run to run without anything meaningful having changed. We therefore also report the **approximation ratio** ($-\langle H \rangle$ divided by the optimal cut value) and the **probability of sampling an optimal cut**. Those are the quantities Adapt-QAOA actually improves.


In [ ]:
thetas_scan = np.linspace(-np.pi, np.pi, 60)
costs_scan = []
for t in thetas_scan:
    e = cudaq.observe(
        kernel_p1_plus_operator, H_hw, n_q,
        edge_src, edge_tgt, edge_w, gamma_p1, beta_p1, [best_op_str], t
    ).expectation()
    costs_scan.append(e)

opt_idx = np.argmin(costs_scan)
opt_extra_theta = thetas_scan[opt_idx]
opt_cost = costs_scan[opt_idx]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(thetas_scan, costs_scan, color='#76b900', linewidth=2,
        label=f'C(θ) for {best_op_label}')
ax.axhline(base_cost, color='#888', linestyle='--', linewidth=1,
           label=f'p=1 baseline ({base_cost:.2f})')
ax.axvline(0, color='#555', linestyle=':', alpha=0.4)
ax.plot(opt_extra_theta, opt_cost, 'o', color='#0074df', markersize=9,
        zorder=5, label=f'Optimum θ={opt_extra_theta:.2f} (C={opt_cost:.2f})')
ax.set_xlabel('θ (operator strength)')
ax.set_ylabel('⟨H⟩ (cost)')
ax.set_title(f'Cost landscape for adding {best_op_label} after QAOA p=1')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

counts_extended_sim = cudaq.sample(
    kernel_p1_plus_operator, n_q,
    edge_src, edge_tgt, edge_w, gamma_p1, beta_p1, [best_op_str], opt_extra_theta,
    shots_count=5000
)

ext_bs = counts_extended_sim.most_probable()
ext_cut = cut_value_from_bitstring(ext_bs)

ext_p_optimal = probability_of_optimal_cut(counts_extended_sim)

print(f"Optimized θ = {opt_extra_theta:.4f}")
print(f"Cost:                {base_cost:.4f} → {opt_cost:.4f}  "
      f"(ΔC = {opt_cost - base_cost:+.4f})")
print(f"Approximation ratio: {-base_cost / max_cut_value:.4f} → "
      f"{-opt_cost / max_cut_value:.4f}")
print(f"P(optimal cut):      {sim_p_optimal:.1%} → {ext_p_optimal:.1%}")
print(f"Most probable bitstring: {ext_bs}  (cut = {ext_cut} / {max_cut_value})")

### 1.8.2 Test your improved circuit on the QPU

You've found the best operator to add and optimized its parameter — all in simulation. Now send the improved circuit to a real QPU and see if hardware confirms the improvement.

Good options for this notebook include **IQM Garnet** (20 qubits), **IQM Emerald** (54 qubits), or a **Rigetti** device available through qBraid. Device availability, queue status, and pricing change over time — before you submit a job, check what is currently online on the [IQM Resonance](https://resonance.meetiqm.com/) portal and/or the [qBraid device catalog](https://account.qbraid.com/devices), and check current QPU rates on the [qBraid pricing page](https://www.qbraid.com/pricing). Then use a machine ID that matches what you see in the catalog.

The notebook keeps paid hardware execution in one clearly labeled cell. Configure that cell, set `RUN_QPU = True`, and run it once. A one-shot guard prevents accidental resubmission, and the cell switches the active target back to `qpp-cpu` afterward. The following CPU and comparison cells are safe to rerun because they never submit QPU jobs.

**Option A — IQM Resonance (direct IQM target).**  
Use CUDA-Q's built-in `iqm` target. Create an API token in the [IQM Resonance](https://resonance.meetiqm.com/) portal, set `IQM_TOKEN`, and point at your Resonance server URL for Garnet or Emerald:

```python
# export IQM_TOKEN="..."   # from your IQM Resonance account profile
iqm_url = "https://<IQM Server>/"  # Garnet or Emerald Resonance URL; or set IQM_SERVER_URL instead
cudaq.set_target("iqm", url=iqm_url)
```

**Option B — QPU through qBraid.**  
qBraid brokers access to IQM, Rigetti, and other vendors through CUDA-Q's `qbraid` target. This path requires **CUDA-Q 0.15.1 or higher**. If needed, upgrade CUDA-Q, then **restart the kernel** and re-run the import cells:

```bash
pip install --upgrade "cudaq>=0.15.1"
```

**Getting a qBraid API key:** sign in at [account.qbraid.com](https://account.qbraid.com), open **Account → API Keys**, and create a key ([qBraid API Keys docs](https://docs.qbraid.com/v2/account/api-keys)). Copy it when shown — it is displayed only once. Then either export it or pass it inline:

```bash
export QBRAID_API_KEY="your_api_key"
```

Pick one currently available device from the [qBraid device catalog](https://account.qbraid.com/devices). Common IDs include:

```python
# IQM Garnet (20 qubits) or Emerald (54 qubits):
cudaq.set_target("qbraid", machine="aws:iqm:qpu:garnet")
# cudaq.set_target("qbraid", machine="aws:iqm:qpu:emerald")
#
# Rigetti via qBraid (confirm the exact ID in the catalog):
# cudaq.set_target("qbraid", machine="aws:rigetti:qpu:ankaa-3")
# cudaq.set_target("qbraid", machine="aws:rigetti:qpu:cepheus-1-108q")
#
# Or, instead of the env var:
# cudaq.set_target("qbraid", machine="aws:iqm:qpu:garnet", api_key="your_api_key")
```

Docs: [CUDA-Q qBraid target](https://nvidia.github.io/cuda-quantum/latest/using/backends/cloud/qbraid.html) · [CUDA-Q IQM / Resonance](https://nvidia.github.io/cuda-quantum/latest/using/backends/hardware/superconducting.html).

**Shot count:** the CPU comparison uses **5000 shots**. The QPU job uses **50 shots** to keep hardware costs down. That is enough to get a sample coloring, but it is much noisier than the simulator — treat it as a quick hardware check, not a matched statistical comparison. Raise `QPU_SHOTS` toward 500–5000 only if you need more stable hardware statistics and can afford the cost.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Exercise 7 (continued):</span>**

1. Run the **CPU-only comparison** first; it shows the p=1 baseline beside p=1 plus your adaptive operator, both with 5000 simulated shots.
2. If you have checked availability and pricing, configure and run the dedicated **QPU submission** cell once with 50 shots.
3. Run the separate **CPU vs. QPU comparison** cell. It only reads the saved QPU counts and cannot submit another hardware job.

**Think about:**
- Does the adaptive operator improve the CPU result?
- How does the cached QPU result differ from simulation? Separate **device noise** from the much smaller QPU shot count.
- What would happen if you repeated the adaptive process by selecting another operator?

</div>


#### CPU-only comparison — safe to rerun

Run this comparison before using hardware. It samples only the local `qpp-cpu` simulator and displays two panels:

1. the standard p=1 QAOA baseline, and
2. p=1 QAOA plus the adaptive operator you selected.

Both use 5000 shots, so this is the fair comparison for judging whether the adaptive operator helped. This cell never contacts a QPU.

In [ ]:
# ══ CPU-ONLY COMPARISON — SAFE TO RERUN ═══════════════════════════════════
SIM_SHOTS = 5000
cudaq.set_target("qpp-cpu")

counts_p1_ref = cudaq.sample(
    kernel_p1_plus_operator,
    n_q,
    edge_src,
    edge_tgt,
    edge_w,
    gamma_p1,
    beta_p1,
    [identity_word],
    0.0,
    shots_count=SIM_SHOTS,
)
p1_ref_bs = counts_p1_ref.most_probable()
p1_ref_cut = cut_value_from_bitstring(p1_ref_bs)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
titles = [
    f"QAOA p=1 (CPU, {SIM_SHOTS} shots)\ncut = {p1_ref_cut} / {max_cut_value}",
    f"p=1 + {best_op_label} (CPU, {SIM_SHOTS} shots)\ncut = {ext_cut} / {max_cut_value}",
]
bitstrings = [p1_ref_bs, ext_bs]

for ax, title, bitstring in zip(axes, titles, bitstrings):
    colors = [
        gray if int(bitstring[node_idx_hw[node]]) == 0 else green
        for node in sampleGraph
    ]
    cut_edges = [
        (u, v)
        for u, v in sampleGraph.edges()
        if bitstring[node_idx_hw[u]] != bitstring[node_idx_hw[v]]
    ]
    nx.draw_networkx_edges(
        sampleGraph,
        pos,
        edgelist=cut_edges,
        width=6,
        alpha=0.4,
        edge_color=green,
        ax=ax,
    )
    nx.draw(sampleGraph, with_labels=True, pos=pos, node_color=colors, ax=ax)
    ax.set_title(title, fontsize=10)

plt.suptitle(
    "CPU comparison: effect of one adaptive operator",
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

base_p_optimal = probability_of_optimal_cut(counts_p1_ref)
ext_p_optimal = probability_of_optimal_cut(counts_extended_sim)

print("\nCPU-only summary (matched 5000-shot budget):")
print(
    f"  QAOA p=1:              cost = {base_cost:6.2f}   "
    f"ratio = {-base_cost / max_cut_value:.3f}   "
    f"P(optimal) = {base_p_optimal:5.1%}   "
    f"sampled cut = {p1_ref_cut} / {max_cut_value}"
)
print(
    f"  p=1 + {best_op_label}:  cost = {opt_cost:6.2f}   "
    f"ratio = {-opt_cost / max_cut_value:.3f}   "
    f"P(optimal) = {ext_p_optimal:5.1%}   "
    f"sampled cut = {ext_cut} / {max_cut_value}"
)

In [ ]:
# ══ PAID QPU SUBMISSION — RUN ONCE ═════════════════════════════════════════
# This is the only cell in the notebook that can submit a paid hardware job.
# Check availability and pricing before changing RUN_QPU to True.
RUN_QPU = False
QPU_SHOTS = 50
QPU_PROVIDER = "qbraid"  # "qbraid" or "iqm"
QPU_MACHINE = "aws:iqm:qpu:garnet"  # Replace with an available qBraid machine ID.
IQM_URL = "https://<IQM Server>/"  # Used only when QPU_PROVIDER == "iqm".
REVERSE_QPU_BITS = QPU_PROVIDER == "iqm"

if not RUN_QPU:
    print("QPU submission is disabled. Set RUN_QPU = True only when ready to submit once.")
elif globals().get("_qpu_job_submitted", False):
    print("QPU submission blocked: this kernel has already submitted the job.")
    print("Use the cached counts_ext_qpu result below; do not rerun the paid job.")
else:
    _qpu_job_submitted = True
    try:
        if QPU_PROVIDER == "qbraid":
            cudaq.set_target("qbraid", machine=QPU_MACHINE)
        elif QPU_PROVIDER == "iqm":
            cudaq.set_target("iqm", url=IQM_URL)
        else:
            raise ValueError("QPU_PROVIDER must be 'qbraid' or 'iqm'.")

        counts_ext_qpu = cudaq.sample(
            kernel_p1_plus_operator,
            n_q,
            edge_src,
            edge_tgt,
            edge_w,
            gamma_p1,
            beta_p1,
            [best_op_str],
            opt_extra_theta,
            shots_count=QPU_SHOTS,
        )
        ext_qpu_bs = counts_ext_qpu.most_probable()
        if REVERSE_QPU_BITS:
            ext_qpu_bs = ext_qpu_bs[::-1]
        ext_qpu_cut = cut_value_from_bitstring(ext_qpu_bs)
        print(f"QPU result ({QPU_SHOTS} shots): {ext_qpu_bs}  "
              f"(cut = {ext_qpu_cut} / {max_cut_value})")
    except Exception:
        _qpu_job_submitted = False
        raise
    finally:
        # Keep every later sample call safely on the local CPU simulator.
        cudaq.set_target("qpp-cpu")



#### CPU vs. QPU comparison — no hardware submission

Run this cell only after the dedicated QPU cell has produced `counts_ext_qpu`. It displays three panels: the CPU baseline, the adaptive CPU result, and the cached adaptive QPU result.

This comparison cell does **not** call `cudaq.sample`; rerunning it cannot submit another QPU job. Remember that the CPU panels use 5000 shots while the QPU panel uses only 50, so use the QPU panel as a qualitative hardware check.

In [ ]:
# ══ CPU VS. CACHED QPU RESULT — NO SAMPLE CALLS ═══════════════════════════
if "counts_ext_qpu" not in globals():
    print("No cached QPU result yet. Run the guarded QPU submission cell once,")
    print("then rerun this comparison cell.")
else:
    qpu_shots_used = globals().get("QPU_SHOTS", 50)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    titles = [
        f"QAOA p=1 (CPU, {SIM_SHOTS} shots)\ncut = {p1_ref_cut} / {max_cut_value}",
        f"p=1 + {best_op_label} (CPU, {SIM_SHOTS} shots)\n"
        f"cut = {ext_cut} / {max_cut_value}",
        f"p=1 + {best_op_label} (QPU, {qpu_shots_used} shots)\n"
        f"cut = {ext_qpu_cut} / {max_cut_value}",
    ]
    bitstrings = [p1_ref_bs, ext_bs, ext_qpu_bs]

    for ax, title, bitstring in zip(axes, titles, bitstrings):
        colors = [
            gray if int(bitstring[node_idx_hw[node]]) == 0 else green
            for node in sampleGraph
        ]
        cut_edges = [
            (u, v)
            for u, v in sampleGraph.edges()
            if bitstring[node_idx_hw[u]] != bitstring[node_idx_hw[v]]
        ]
        nx.draw_networkx_edges(
            sampleGraph,
            pos,
            edgelist=cut_edges,
            width=6,
            alpha=0.4,
            edge_color=green,
            ax=ax,
        )
        nx.draw(sampleGraph, with_labels=True, pos=pos, node_color=colors, ax=ax)
        ax.set_title(title, fontsize=10)

    plt.suptitle(
        "CPU simulation vs. cached QPU result",
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    print(
        f"QPU sampled cut: {ext_qpu_cut} / {max_cut_value} "
        f"from {qpu_shots_used} shots."
    )
    print(
        f"CPU panels use {SIM_SHOTS} shots, so do not over-interpret "
        "CPU–QPU differences."
    )

By adding just **one** carefully chosen operator, you've taken the first step of the Adapt-QAOA algorithm by hand. Even when the single most probable bitstring doesn't change (or briefly looks worse), a drop in ⟨H⟩ and a rise in the probability of an optimal cut are the signals that *adaptive* circuit construction is investing depth where it helps.

Notice how this connects to the widget:
- The **gradient** you computed is exactly the slope of the C(θ) curve at θ = 0 that the widget visualized on the triangle.
- The **C(θ) graph** you plotted is the same curve the widget draws — just computed with CUDA-Q on the full 7-node graph instead of a 3-node toy example.

Repeating this process — evaluate gradients → pick the best operator → optimize → repeat — is the full Adapt-QAOA loop. Each iteration adds one more targeted operator, steadily driving the cost toward the optimum without the barren plateau problem that plagues deep fixed-structure circuits.

In the next section, we'll see how **QAOA-GPT** bypasses this iterative process entirely by using AI to generate the full circuit in a single forward pass.


---

## 1.9 Addressing circuit depth with AI: QAOA-GPT

Adapt-QAOA tackles the barren plateau problem by cleverly choosing which operators to add. **QAOA-GPT** takes this idea a step further: what if, instead of running the optimization loop at all, we could use AI to *directly generate* good QAOA circuits and parameters?

**QAOA-GPT** is a framework that does exactly this. It uses a Generative Pretrained Transformer (GPT) — the same family of models behind ChatGPT — trained on data generated by Adapt-QAOA runs. Given a new graph, the model generates QAOA circuit parameters in a single forward pass, bypassing the expensive iterative optimization entirely.

<img src="https://github.com/NVIDIA/cuda-q-academic/blob/main/images/QAOA-GPT-flowchart.png?raw=true" alt="Flowchart of QAOA-GPT - Adapt-QAOA data is used to train a GPT model, which then generates QAOA circuits and parameters for new graphs in a single forward pass." />

The flowchart above summarizes how QAOA-GPT learns from Adapt-QAOA-generated training data and then predicts good circuits directly for new graphs.

### How it works

1. **Training data generation:** Adapt-QAOA is run on many different graphs to produce high-quality circuits and parameters. The graph structure and resulting circuit are encoded as token sequences.
2. **Model training:** A GPT model is trained on these sequences, learning the relationship between graph features and optimal QAOA parameters.
3. **Inference:** Given a new, unseen graph, the trained model generates circuit parameters in one shot — no quantum hardware or optimization loop required.

### Published results

Experiments from the original paper (Tyagin *et al.*, 2025) show that QAOA-GPT **closely matches Adapt-QAOA** while being orders of magnitude faster at inference time:

| | **QAOA-GPT** (best of 5) | **Adapt-QAOA** | **Standard QAOA** (p=n) |
|:--|--:|--:|--:|
| 10-node AR | 0.971 ± 0.009 | 0.974 ± 0.003 | 0.926 ± 0.108 |
| 12-node AR | 0.971 ± 0.009 | 0.973 ± 0.002 | 0.934 ± 0.108 |
| 14-node AR | 0.972 ± 0.007 | 0.973 ± 0.002 | 0.946 ± 0.102 |

*(Table I from the paper — mean approximation ratio over 1000 random weighted Erdős–Rényi graphs per size)*

The key advantage is **scalability**: QAOA-GPT's inference time stays nearly constant as the problem size grows (it's a single neural network forward pass), while Adapt-QAOA's runtime grows rapidly due to gradient-based operator selection at each layer. Training data generation (running Adapt-QAOA on many graphs) is a one-time cost that can be GPU-accelerated with CUDA-Q.

A follow-up paper (Sunny *et al.*, 2025) extends QAOA-GPT to **higher-order optimization problems** (HUBO with cubic interaction terms) on 8- and 16-qubit heavy-hex lattices, achieving average approximation ratios exceeding 0.95 — demonstrating that the approach generalizes beyond Max Cut.

**Further reading:**
- Tyagin *et al.*, ["QAOA-GPT: Efficient Generation of Adaptive and Regular QAOA Circuits"](https://arxiv.org/abs/2504.16350) (2025)
- [QAOA-GPT source code](https://github.com/marwafar/cuda-quantum/tree/QAOA-GPT-v1/docs/sphinx/applications/python/qaoa_gpt_src)
- Sunny *et al.*, ["Extending QAOA-GPT to Higher-Order Quantum Optimization Problems"](https://arxiv.org/abs/2511.07391) (2025)


---

## 1.10 Take-home challenge: compare QAOA variants on a weighted graph

You've now seen three approaches to quantum optimization: **standard QAOA**, **Adapt-QAOA**, and **QAOA-GPT**. It's time to put them head-to-head.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #1a1a1a;">

**<span style="color: #487a00; font-size: 1.17em;">Challenge:</span>**

1. Create a random **8-vertex weighted** graph (e.g. using `nx.erdos_renyi_graph` with random edge weights).
2. Run **standard QAOA** for the weighted graph at depths p = 1 and p = 4 using `optimize_maxcut_qaoa` — the helper incorporates each graph edge's `weight` attribute in both the Hamiltonian and circuit.
3. Run **Adapt-QAOA** by extending the one-step implementation from Section 1.8.1 into a repeated operator-selection loop for the weighted graph; the helper's `maxcut_problem` already carries each edge's weight into both the Hamiltonian and the circuit.
4. Run **QAOA-GPT** using the source code at:  
   [QAOA-GPT source code](https://github.com/marwafar/cuda-quantum/tree/QAOA-GPT-v1/docs/sphinx/applications/python/qaoa_gpt_src)
5. Compare **approximation ratio**, **number of parameters**, and **estimated two-qubit gate count** across all three methods.

</div>

**Questions to consider:**
- Which method reaches the best approximation ratio? At what cost in circuit depth?
- How do the Adapt-QAOA and QAOA-GPT flowcharts (shown earlier in this notebook) reflect their different trade-offs between optimization effort and circuit quality?


---

## Conclusion

In this tutorial, you:
- Defined the **Max Cut** problem and understood why it's NP-hard to solve classically.
- Learned core quantum concepts — qubits, superposition, measurement, and quantum interference.
- Saw how a graph is translated into a quantum **Hamiltonian** and implemented **QAOA** with CUDA-Q (via a reusable helper, then an explicit kernel).
- Ran your QAOA circuit on a **real quantum processor** (IQM Garnet/Emerald or Rigetti via IQM Resonance or qBraid) and verified that it agrees with simulation.
- Explored the **circuit-depth** challenge in scaling QAOA and two responses to it:
    - **Adapt-QAOA** — adaptive circuit construction that adds only useful operators.
    - **QAOA-GPT** — AI-generated circuits and parameters in a single forward pass.

Quantum computing is a rapidly evolving field, and the intersection of AI and quantum algorithm design is one of its most exciting frontiers. We hope this tutorial has given you both a practical introduction and a sense of where the field is heading.

If you'd like to keep exploring, try the **take-home challenge** in Section 1.10, which puts all three QAOA variants to the test on a weighted graph you design yourself.



**Related Notebooks:**
* [QAOA for Max Cut: 01 Max Cut with QAOA](https://github.com/NVIDIA/cuda-q-academic/blob/main/qaoa-for-max-cut/01_Max-Cut-with-QAOA.ipynb) — foundational QAOA implementation details used by this workshop.
* [AI for Quantum: Compiling Unitaries Using Diffusion Models](https://github.com/NVIDIA/cuda-q-academic/blob/main/ai-for-quantum/01_compiling_unitaries_diffusion.ipynb) — complementary AI-for-quantum workflow in the same track family.
